Setup project paths

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
import json

ROOT = Path(r"E:\S T A G E\Project")
OUT = ROOT / "outputs" / "SRC002"
OUT.mkdir(parents=True, exist_ok=True)

url = "https://www.carenity.com/forum/endometriose-347"

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

print("Status:", resp.status_code)
print("Final URL:", resp.url)

html = resp.text
print("HTML length:", len(html))

soup = BeautifulSoup(html, "lxml")

# Save raw HTML too (very useful for debugging)
(OUT / "page_raw.html").write_text(html, encoding="utf-8")

print("Saved raw HTML to:", OUT / "page_raw.html")

Status: 200
Final URL: https://www.carenity.com/forum/endometriose-347
HTML length: 107796
Saved raw HTML to: E:\S T A G E\Project\outputs\SRC002\page_raw.html


In [2]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json

soup = BeautifulSoup(html, "lxml")

threads = []

# Doctissimo old forum pages usually contain topic links with .htm hrefs
for a in soup.select("a[href]"):
    href = a.get("href", "").strip()
    text = a.get_text(" ", strip=True)

    if not href or not text:
        continue

    full_url = urljoin(url, href)

    # Keep only forum topic links related to subjects
    if ".htm" in href and "liste_sujet" not in href:
        threads.append({
            "title": text,
            "url": full_url
        })

# Deduplicate
seen = set()
clean_threads = []
for t in threads:
    key = (t["title"], t["url"])
    if key not in seen:
        seen.add(key)
        clean_threads.append(t)

print("Threads found:", len(clean_threads))
clean_threads[:10]

Threads found: 0


[]

In [3]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import re

soup = BeautifulSoup(html, "lxml")

threads = []

for a in soup.select("a[href]"):
    href = a.get("href", "").strip()
    text = a.get_text(" ", strip=True)

    if not href or not text:
        continue

    full_url = urljoin(url, href)

    # Keep only forum thread links
    if re.search(r"endometriose.*sujet_", href, re.I):
        threads.append({
            "title": text,
            "url": full_url
        })

# Remove duplicates
seen = set()
clean_threads = []
for t in threads:
    key = (t["title"], t["url"])
    if key not in seen:
        seen.add(key)
        clean_threads.append(t)

print("Real thread links found:", len(clean_threads))
clean_threads[:10]

Real thread links found: 0


[]

In [4]:
out_file = OUT / "posts_clean.jsonl"

with out_file.open("w", encoding="utf-8") as f:
    for item in clean_threads:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved clean threads:", out_file)
print("Count:", len(clean_threads))

Saved clean threads: E:\S T A G E\Project\outputs\SRC002\posts_clean.jsonl
Count: 0


In [5]:
lines = (OUT / "posts.jsonl").read_text(encoding="utf-8").splitlines()
print("Lines:", len(lines))

if lines:
    print(lines[0])

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\S T A G E\\Project\\outputs\\SRC002\\posts.jsonl'

In [33]:
first_thread = clean_threads[0]["url"]
print(first_thread)

https://forum.doctissimo.fr/sante/Endometriose/endometriose-sujet_7362_1.htm


In [34]:
resp2 = requests.get(first_thread, headers=headers, timeout=30)
resp2.raise_for_status()

thread_html = resp2.text
thread_soup = BeautifulSoup(thread_html, "lxml")

print("Thread URL:", first_thread)
print("Thread page size:", len(thread_html))
print(thread_soup.title.get_text(" ", strip=True) if thread_soup.title else "No title")

Thread URL: https://forum.doctissimo.fr/sante/Endometriose/endometriose-sujet_7362_1.htm
Thread page size: 413809
L'endométriose - Endométriose - FORUM Santé - Club.doctissimo.fr
